# NYC Flights 2013 - Business Questions

### PART III


In [1]:
import pandas as pd
import numpy as np
import nycflights13 as nyc


flights_o = nyc.flights
airlines_o = nyc.airlines
airports_o = nyc.airports
planes_o = nyc.planes
weather_o = nyc.weather



C:\Users\anupr\AppData\Roaming\Python\Python314\site-packages\nycflights13\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


## Setup — feature engineering/ data cleaning

Originally carried out in Part 1 notebook - recreated here so this notebook is also self-contained.

In [2]:
##**Flights**

flights= flights_o.copy()

#convert main datetime-related columns safely
date_cols=["time_hour"]
for col in date_cols:
    flights[col] = pd.to_datetime(flights[col], errors="coerce")

#replace any empty fields with NaN (not a number)
flights["tailnum"]=flights["tailnum"].replace("",np.nan)

#remove exact duplicates
flights = flights.drop_duplicates().reset_index(drop=True) #reset the column numbers to 1,2,3,...

print("Cleaned Shape:",flights.shape)
#print(flights)

###IMP NOTE: NO further FE
#`dep_time`, `dep_delay`, `arr_time`, `arr_delay`, and `air_time` are  missing
#for the same ~8,255 rows — these are cancelled flights, not bad data. 
#`tailnum` is missing for a smaller set (2,512 rows), most likely older/decommissioned aircraft not in FAA records. 

## **Weather**

weather=weather_o.copy()
date_cols = ["time_hour"]
for col in date_cols:
    weather_o[col] = pd.to_datetime(weather_o[col], errors="coerce")
    
#2. clean wind speed outlier value
weather = weather.drop(weather[weather["wind_speed"] > 1000].index)

#3. dropping wind gust column altogether 
weather = weather.drop(columns=["wind_gust"])

#display(weather.head(5))

#print("Cleaned shape:", weather.shape)


###**Planes**
planes= planes_o.copy()
planes = planes.drop(columns=["speed"])

#display(planes.head(5))

#print("Cleaned shape:", planes.shape)

###**Airport**
airports= airports_o.copy()

#display(airport.head(5))

#print("Cleaned shape:", airport.shape)

###**Airlines**
airlines= airlines_o.copy()

#display(airlines.head(5))

#print("Cleaned shape:", airlines.shape)


Cleaned Shape: (336776, 19)


## Q1. Flight Operations - which airline operates the largest share of flights from NYC?

In [3]:
market_share=flights.groupby("carrier").size().sort_values(ascending=False)
market_share_pct=(market_share / market_share.sum() * 100).round(2)

q1=market_share_pct.to_frame("pct_share").merge(airlines, on="carrier").set_index("name")
q1["n_flights"]=market_share.reindex(q1["carrier"]).values

print(q1[["carrier","n_flights", "pct_share"]].head(5))

                         carrier  n_flights  pct_share
name                                                  
United Air Lines Inc.         UA      58665      17.42
JetBlue Airways               B6      54635      16.22
ExpressJet Airlines Inc.      EV      54173      16.09
Delta Air Lines Inc.          DL      48110      14.29
American Airlines Inc.        AA      32729       9.72


**Answer:** United Airlines (UA) is the clear market leader - 58,665 flights, 17.4% of all
NYC departures in 2013. JetBlue (16.2%) and ExpressJet (16.1%) are close behind

## Q2. Delay Reduction - which airline has the worst average departure delay?

In [4]:
worst_dep=flights.groupby("carrier")["dep_delay"].mean().sort_values(ascending=False).round(2)

worst_dep=worst_dep.to_frame("avg_dep_delay").merge(airlines, on="carrier").set_index("name")

print(worst_dep.head(5))

                            carrier  avg_dep_delay
name                                              
Frontier Airlines Inc.           F9          20.22
ExpressJet Airlines Inc.         EV          19.96
Mesa Airlines Inc.               YV          19.00
AirTran Airways Corporation      FL          18.73
Southwest Airlines Co.           WN          17.71


**Answer:** Frontier (F9) has the worst average departure delay (20 min), with ExpressJet (EV) and Mesa (YV) close behind.

## Q3. Customer Experience - which airline has the highest average arrival delay?

In [5]:
worst_arr=flights.groupby("carrier")["arr_delay"].mean().sort_values(ascending=False).round(2)

worst_arr=worst_arr.to_frame("avg_arr_delay").merge(airlines, on="carrier").set_index("name")

print(worst_arr.head(5))

                            carrier  avg_arr_delay
name                                              
Frontier Airlines Inc.           F9          21.92
AirTran Airways Corporation      FL          20.12
ExpressJet Airlines Inc.         EV          15.80
Mesa Airlines Inc.               YV          15.56
SkyWest Airlines Inc.            OO          11.93


**Answer:** Frontier (F9) again tops the list on arrival delay (22 min avg), followed by AirTran and ExpressJet. The same four carriers (Frontier, ExpressJet, Mesa, AirTran) appear in the worst four for both departure and arrival delay, which suggests the carriers that start late also arrive late.

## Q4. Airport Operations - which airport handles the highest number of departures?

In [6]:
dep_airport=flights.groupby("origin").size().sort_values(ascending=False)

print(dep_airport)

origin
EWR    120835
JFK    111279
LGA    104662
dtype: int64


**Answer:** EWR handles the most departures, narrowly ahead of JFK and LGA 104,662. 

## Q5. Route Optimization - top 10 busiest routes

In [7]:
fr=flights.copy()

fr["route"]=fr["origin"]+" to "+fr["dest"] #creating the route column based on origin n dest columns

b_routes=fr.groupby("route").size().sort_values(ascending=False).head(10)
print(b_routes)

route
JFK to LAX    11262
LGA to ATL    10263
LGA to ORD     8857
JFK to SFO     8204
LGA to CLT     6168
EWR to ORD     6100
JFK to BOS     5898
LGA to MIA     5781
JFK to MCO     5464
EWR to BOS     5327
dtype: int64


**Answer:** JFK to LAX is the busiest route by a wide margin.

## Q6. Seasonal Planning - which month has the highest flight volume?

In [8]:
month_vol=flights.groupby("month").size().sort_values(ascending=False)

print(month_vol.head(3))

month
7     29425
8     29327
10    28889
dtype: int64


**Answer:** July has the highest flight volume — perhaps peak summer travel time.

## Q7. Peak-Hour Management - which hours have the highest number of departures?

In [9]:
hour_vol= flights.groupby("hour").size().sort_values(ascending=False)

print(hour_vol.head(5))

hour
8     27242
6     25951
17    24426
15    23888
16    23002
dtype: int64


**Answer:** 8am is the busiest scheduled hour.

## Q8. Delay Investigation - which routes have the highest average arrival delay?

In [10]:
route_delay=fr.groupby("route").agg(n=("flight", "count"), avg_arr_delay=("arr_delay", "mean")).round(2)

print(route_delay.sort_values("avg_arr_delay",ascending=False).head(10))

               n  avg_arr_delay
route                          
EWR to CAE   104          44.59
EWR to TYS   323          41.15
EWR to TUL   315          33.66
EWR to OKC   346          30.62
EWR to JAC    23          29.89
EWR to RIC  1667          25.82
EWR to MKE  1094          24.00
EWR to DSM   411          23.55
EWR to MSN   354          23.20
EWR to PWM   770          23.01


**Answer:**  EWR to CAE route tops the list.

## Q9. Airline Benchmarking - flight volume, avg departure delay, avg arrival delay combined

In [11]:
ab=flights.groupby("carrier").agg(
    n_flights=("flight", "count"), avg_dep_delay=("dep_delay", "mean"), avg_arr_delay=("arr_delay", "mean")
).round(2)

ab=ab[ab["n_flights"]>1000]  # exclude carriers too small to benchmark fairly - if this is not done the answer is different
ab["combined_rank"]=(ab["avg_dep_delay"].rank()+ab["avg_arr_delay"].rank())/2
ab=ab.merge(airlines, on="carrier").set_index("name").sort_values("combined_rank")

print(ab[["n_flights","avg_dep_delay","avg_arr_delay","combined_rank"]].head(5))

                        n_flights  avg_dep_delay  avg_arr_delay  combined_rank
name                                                                          
American Airlines Inc.      32729           8.59           0.36            1.5
Delta Air Lines Inc.        48110           9.26           1.64            2.5
US Airways Inc.             20536           3.78           2.13            2.5
Virgin America               5162          12.87           1.76            4.5
United Air Lines Inc.       58665          12.11           3.56            5.0


**Answer:** For carriers with flight count >1000 - American Airlines ranks best overall on the combined departure+arrival delay rank, followed by  Delta and US Airways.
Without the flight count filter - Alaska Airlines ranks best, followed by Hawaiian and US Airways. American Airlines ranks 4th in this scenario.

## Q10. Customer Service - highest % of flights delayed (arrival delay>15 min)

In [12]:
pct15=flights.groupby("carrier")["arr_delay"].apply(lambda x:(x>15).mean()*100).round(2)

pct15=pct15.sort_values(ascending=False).to_frame("pct_delayed_15min").merge(airlines, on="carrier").set_index("name")

print(pct15.head(5))

                            carrier  pct_delayed_15min
name                                                  
Frontier Airlines Inc.           F9              37.08
AirTran Airways Corporation      FL              32.67
ExpressJet Airlines Inc.         EV              29.59
Mesa Airlines Inc.               YV              28.45
JetBlue Airways                  B6              25.83


**Answer:** Frontier (F9) has the highest share of delayed flights by management's definition - 37.1% of its flights arrive more than 15 minutes late.

## Q11. Fuel/Distance Analysis - which airlines operate the longest average-distance flights?

In [13]:
dist_by_carrier=flights.groupby("carrier")["distance"].mean().sort_values(ascending=False).round(2)

dist_by_carrier=dist_by_carrier.to_frame("avg_distance").merge(airlines, on="carrier").set_index("name")

print(dist_by_carrier.head(5))

                       carrier  avg_distance
name                                        
Hawaiian Airlines Inc.      HA       4983.00
Virgin America              VX       2499.48
Alaska Airlines Inc.        AS       2402.00
Frontier Airlines Inc.      F9       1620.00
United Air Lines Inc.       UA       1529.11


**Answer:** Hawaiian Airlines flies by far the longest average distance.

## Q12. Capacity Planning - destinations receiving the highest number of flights

In [14]:
dest_vol=flights.groupby("dest").size().sort_values(ascending=False)

print(dest_vol.head(10))

dest
ORD    17283
ATL    17215
LAX    16174
BOS    15508
MCO    14082
CLT    14064
SFO    13331
FLL    12055
MIA    11728
DCA     9705
dtype: int64


**Answer:** Chicago O'Hare (ORD) and Atlanta (ATL) receive the most flights from NYC.

## Q13. Underperforming Routes - routes with 100 flights but unusually high avg arrival delay

In [15]:
route_delay_100=fr.groupby("route").agg(n=("flight","count"),avg_arr_delay=("arr_delay","mean")).round(2)

route_delay_100=route_delay_100[route_delay_100["n"]>=100].sort_values("avg_arr_delay",ascending=False)

print(route_delay_100.head(10))

               n  avg_arr_delay
route                          
EWR to CAE   104          44.59
EWR to TYS   323          41.15
EWR to TUL   315          33.66
EWR to OKC   346          30.62
EWR to RIC  1667          25.82
EWR to MKE  1094          24.00
EWR to DSM   411          23.55
EWR to MSN   354          23.20
EWR to PWM   770          23.01
JFK to CMH   742          22.92


**Answer:** EWR-CAE (Columbia, SC) still tops the list at 44.6 min avg delay on 104 flights - enough volume to trust the number. EWR-TYS, EWR-TUL, and
EWR-OKC follow all in 30-41 min time range. 
9 of top 10 underperforming route originates at Newark, highlighting that this is an airport-level issue

## Q14. Airport Performance, compare NYC airports on average departure delay - 
## Which airport requires operational improvement?

In [16]:
airport_dep_delay=flights.groupby("origin")["dep_delay"].mean().sort_values(ascending=False).round(2)

print(airport_dep_delay)

origin
EWR    15.11
JFK    12.11
LGA    10.35
Name: dep_delay, dtype: float64


**Answer:** Newark (EWR) requires the most operational attention - 15.1 min average
departure delay vs. JFK's 12.1 and LGA's 10.4. Combined with Q4 (EWR also has the highest
departure volume) and Q13 (top 9 underperforming routes with origin EWR) Newark is the single highest-priority airport for any delay-reduction
initiative.

## Q15. Executive Business Problem - reduce overall delays by 10%. 
## Where should the CEO focus?

In [17]:
print("TOP 5 AIRLINES TO PRIORITIZE (worst avg arr delay):") # calculated in Q3
print(worst_arr.head(5))
print("\nTOP AIRPORT TO PRIORITIZE (with the highest departure delay) :")     #calculated in Q14
print(airport_dep_delay.head(1))      
print("\nTOP 5 ROUTES TO PRIORITIZE (top underperforming routes) :")    #calculated in Q13
print(route_delay_100.head(5))

TOP 5 AIRLINES TO PRIORITIZE (worst avg arr delay):
                            carrier  avg_arr_delay
name                                              
Frontier Airlines Inc.           F9          21.92
AirTran Airways Corporation      FL          20.12
ExpressJet Airlines Inc.         EV          15.80
Mesa Airlines Inc.               YV          15.56
SkyWest Airlines Inc.            OO          11.93

TOP AIRPORT TO PRIORITIZE (with the highest departure delay) :
origin
EWR    15.11
Name: dep_delay, dtype: float64

TOP 5 ROUTES TO PRIORITIZE (top underperforming routes) :
               n  avg_arr_delay
route                          
EWR to CAE   104          44.59
EWR to TYS   323          41.15
EWR to TUL   315          33.66
EWR to OKC   346          30.62
EWR to RIC  1667          25.82


**Executive Summary:**

**Airlines (top 5 by avg arrival delay):** Frontier, AirTran, ExpressJet, Mesa, SkyWest.

The first four also rank in the top 5 for departure delay (Q2) and for share of flights more than 15 minutes late (Q10), so the problem is consistent across every cut of the data. SkyWest appears on arrival delay only.

**Airport:** Newark (EWR)

This airport is the highest-volume origin (Q4) and also the worst-performing on delay (Q14).
Any airport-level fix for EWR is highly recommended to see maximum improvement.

**Routes (top 5, >=100 flights):** EWR-CAE, EWR-TYS, EWR-TUL, EWR-OKC, EWR-RIC 

Though the finding is for Routes, but again the issue highlighted is regarding all Newark airport departures - which is the same signal as the airport-level finding.
